# Builds a Brochure

In [68]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display
from bs4 import BeautifulSoup
import requests


### Web link Finder
Create Web Scraper which can find out all the relevant links of the website.

In [69]:
# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


In [70]:
url = "https://www.mheducation.co.in/"

In [71]:
def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [72]:
links = fetch_website_links(url)

### Web Content Finder
Create Web Scraper which can scrape all the content from given website links.

In [73]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [74]:
# SYSTEM PROMPT FROM SCRAPPING THE LINKS IN THE WEBSITE
system_prompt_for_link = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON (no extra text) as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [75]:
# USER PROMPT TO DECIDE WHICH LINKS ARE RELEVANT FOR THE BROCHURE
def get_user_prompt_for_link(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    user_prompt += "\n".join(links)
    return user_prompt

In [76]:
print(get_user_prompt_for_link(url))


Here is the list of links on the website https://www.mheducation.co.in/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#maincontent
https://www.mheducation.co.in/contact-mcgraw-hill-india
https://www.mheducation.co.in/contact-mcgraw-hill-india
https://www.mheducation.co.in/where-to-buy-mcgraw-hill-india-books
https://www.mheducation.co.in/publish
https://www.mheducation.co.in/contact-mcgraw-hill-india
/blog
https://www.mheducation.co.in/
https://www.mheducation.co.in/test-preparation
https://www.mheducation.co.in/highereducation
https://www.mheducation.co.in/
https://www.mheducation.co.in/about-mcgraw-hill-india
https://www.mheducation.co.in/about-mcgraw-hill-india
https://www.mheducation.co.in/social-responsibility
https://www.mheducation.co.in/copyright-permissions
https://www.mheducation.co.in/divers

### Filter out the usable links for 

In [77]:
MODEL = 'llama3.2'
ollama = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'
)

def generate_relevant_links(url):
    response = ollama.chat.completions.create(model=MODEL, messages=[
                {"role": "system", "content": system_prompt_for_link},
                {"role": "user", "content": get_user_prompt_for_link(url)}
            ])
    relevant_links = response.choices[0].message.content
    try:
        return json.loads(relevant_links)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON: {relevant_links}")
        return {"links": []}

### Fetch Content from all relevant links

In [78]:
def fetch_page_and_all_relevant_links_data(url):
    contents = fetch_website_contents(url)
    relevant_links_data = generate_relevant_links(url)
    
    # Ensure we have a dictionary with 'links' key
    if isinstance(relevant_links_data, str):
        relevant_links_data = json.loads(relevant_links_data)
    
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    
    if 'links' in relevant_links_data:
        for link in relevant_links_data['links']:
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
    
    return result

In [79]:
print(fetch_page_and_all_relevant_links_data(url))

## Landing Page:

Textbooks | Competitive Exams | Professional books | McGraw Hill India

The store will not work correctly when cookies are disabled.
JavaScript seems to be disabled in your browser.
For the best experience on our site, be sure to turn on Javascript in your browser.
Skip to Content
Support & Contact
Contact Us
Where to Buy
Publish With Us
View All
Blog
Toggle Nav
Test Prep
Higher Ed
Professional
About
About McGraw Hill India
Social Responsibility
Copyright & Permissions
Diversity & Inclusion
Piracy
Contact Us
View All
Search
Search
Content Area
Close
Close
Students and educators can easily transition to online learning with
McGraw Hill India eBooks
and
Digital learning solutions
Unlock the Potential
Learning creates endless possibilities. McGraw Hill India empowers educators and students to achieve their goals.
Because learning changes everything.®
Test Prep
Discover effective study materials for your test preparation needs.
Go to Test Prep >
Higher Ed
Discover the rig

# Generate Brochure

In [80]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [81]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links_data(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [83]:
get_brochure_user_prompt("McGraw Hill", url)

'\nYou are looking at a company called: McGraw Hill\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nTextbooks | Competitive Exams | Professional books | McGraw Hill India\n\nThe store will not work correctly when cookies are disabled.\nJavaScript seems to be disabled in your browser.\nFor the best experience on our site, be sure to turn on Javascript in your browser.\nSkip to Content\nSupport & Contact\nContact Us\nWhere to Buy\nPublish With Us\nView All\nBlog\nToggle Nav\nTest Prep\nHigher Ed\nProfessional\nAbout\nAbout McGraw Hill India\nSocial Responsibility\nCopyright & Permissions\nDiversity & Inclusion\nPiracy\nContact Us\nView All\nSearch\nSearch\nContent Area\nClose\nClose\nStudents and educators can easily transition to online learning with\nMcGraw Hill India eBooks\nand\nDigital learning solutions\nUnlock the Potential\nLearning creates

In [84]:
# def create_brochure(company_name, url):
#     response = ollama.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": brochure_system_prompt},
#             {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
#         ],
#     )
#     result = response.choices[0].message.content
#     display(Markdown(result))


def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [85]:
# create_brochure("McGraw Hill", url)
stream_brochure("McGraw Hill", url)

# McGraw Hill: Unleashing Endless Possibilities for Students and Professionals Alike


Welcome to McGraw Hill, a global leader in education and professional development. Our mission is to empower educators and students to achieve their goals by providing high-quality learning solutions that deliver great results.

## A Culture of Innovation and Support


At McGraw Hill, we're passionate about creating education solutions that make a real difference in the lives of our customers. We believe that every educator teaches differently, and that every institution has a unique approach that makes it distinct. Our employees are just as unique, and that's what makes us so innovative.


## Empowering Educators and Students


We don't just aim to help students reach their full potential - we want to ensure that our employees do too. That's why we offer flexible career paths, diverse work experiences, and training opportunities that will take your skills to the next level.


Our customers trust us for top-notch learning solutions across various subjects:


*   **Test Prep**: Unlock effective study materials and strategies for test preparation.
*   **Higher Education**: Discover cutting-edge course materials designed for post-secondary institutions.
*   **Professional Development**: Find credible training resources on topics like medical, business, technology, and more.

## Join Our Team


Ready to take the next step in your career path? Browse our open job listings and join a diverse team of professionals who share your passion for learning:

[View Job Openings](careers-page)


Don't just teach differently - work with a team that values and celebrates individuality. Connect with us on social media or visit our blog to learn more about how we're shaping the future of education.

# **Follow Us**

**Social Responsibility**
Discover our initiatives aimed at making a positive impact on communities around the world:

*   [Learn More](social-responsibility)

#